# EEGText D0 — corpus inventory and audit

This notebook performs the download-free first milestone. It inventories the official ZuCo OSF release, saves exact remote file metadata to Google Drive, and audits any task directories that are already present. It does **not** download EEG files or load a model.

Run the cells in order. Only the Drive paths in Cell 2 may need editing.

In [ ]:
# 1) Fetch the project and run the download-free unit tests.
from pathlib import Path
import json, os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)

EEGTEXT_ROOT = PROJECT_ROOT / "eegtext"
os.chdir(EEGTEXT_ROOT)
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])
print("EEGText source and tests are ready.")

In [ ]:
# 2) Mount Drive and declare data locations. Edit only these paths if needed.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
LABELS_CSV = THESIS_ROOT / "Data/zuco_sentiment_labels_task1_fixed.csv"
TASK_DIRS = {
    "SR": THESIS_ROOT / "Data/zuco_og_raw",
    "NR": THESIS_ROOT / "Data/zuco_1_task2_nr",
    "TSR": THESIS_ROOT / "Data/zuco_1_task3_tsr",
}
MANIFEST_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/eegtext/corpus_manifests"
INVENTORY_ROOT = MANIFEST_ROOT / "official_osf_zuco_1"
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
print("Task directories:")
for task, path in TASK_DIRS.items():
    print(f"  {task}: {path} ({'present' if path.exists() else 'missing'})")
print("Labels:", LABELS_CSV, "(present)" if LABELS_CSV.exists() else "(missing)")

In [ ]:
# 3) Inventory official ZuCo 1.0 OSF metadata. This downloads metadata only.
ZUCO_1_OSF_NODE = "q3zws"
run([
    sys.executable, "run.py", "inventory-osf",
    "--node", ZUCO_1_OSF_NODE,
    "--output-dir", str(INVENTORY_ROOT),
])
inventory = json.loads((INVENTORY_ROOT / "inventory.json").read_text())
known_gib = inventory["total_known_size_bytes"] / (1024 ** 3)
print(f"Inventoried {inventory['file_count']:,} files; known total size: {known_gib:.2f} GiB")
print("Saved:", INVENTORY_ROOT / "files.csv")
print()
print("Candidate MATLAB paths containing task markers:")
shown = 0
for item in inventory["files"]:
    path = item["path"]
    if path.lower().endswith(".mat") and any(marker in path.upper() for marker in ("SR", "NR", "TSR")):
        print(f"  {item['size_bytes']!s:>12}  {path}")
        shown += 1
        if shown == 40:
            print("  ... first 40 candidates shown; inspect files.csv for the complete list")
            break

In [ ]:
# 4) Audit every task directory currently present in Drive.
TASK_PATTERNS = {
    "SR": "results*_SR.mat",
    "NR": "results*_NR.mat",
    "TSR": "results*_TSR.mat",
}
summaries = {}
manifest_paths = []
for task, raw_dir in TASK_DIRS.items():
    if not raw_dir.exists():
        print(f"Skipping {task}: directory is not present yet: {raw_dir}")
        continue
    output_dir = MANIFEST_ROOT / f"zuco_1_{task.lower()}"
    command = [
        sys.executable, "run.py", "audit-zuco",
        "--raw-dir", str(raw_dir),
        "--dataset", "zuco",
        "--release", "1.0",
        "--task", task,
        "--pattern", TASK_PATTERNS[task],
        "--output-dir", str(output_dir),
    ]
    if task == "SR" and LABELS_CSV.exists():
        command.extend(["--labels-csv", str(LABELS_CSV)])
    run(command)
    manifest_path = output_dir / "recordings.csv"
    manifest_paths.append(manifest_path)
    summaries[task] = json.loads((output_dir / "summary.json").read_text())

if manifest_paths:
    combined_dir = MANIFEST_ROOT / "zuco_1_combined"
    combine_command = [sys.executable, "run.py", "combine-manifests"]
    for manifest_path in manifest_paths:
        combine_command.extend(["--manifest", str(manifest_path)])
    combine_command.extend(["--output-dir", str(combined_dir)])
    run(combine_command)
    summaries["combined"] = json.loads((combined_dir / "summary.json").read_text())

print()
print("Completed audit summaries:")
print(json.dumps(summaries, indent=2))
print()
print("No raw EEG file was created, moved, or modified by this notebook.")

## What to save after this run

The inventory and audit reports are already persistent in Drive. Share the printed inventory summary and the relevant rows from `official_osf_zuco_1/files.csv` before selecting the large Task 2/3 downloads. The next milestone will lock exact file paths, total bytes, and a resumable Drive-only download command.